In [ ]:
import blocks
import torch
import torch.nn as nn

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
class GPT(nn.Module):
    """ One layer of GPT with attention and FFNN """

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.attention = blocks.MaskedMultiHeadAttention(d_model, num_heads)
        self.ffnn = blocks.FFNN(d_model, hidden_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of x: (Batch_size, num_tokens, d_model)
        x = x + self.dropout(self.attention(self.norm1(x))) # Residual connection with Pre Normalization
        x = x + self.dropout(self.ffnn(self.norm2(x))) # Residual connection with Pre Normalization
        # Dimension: (Batch_size, num_tokens, d_model)
        return x

In [ ]:
class GPTStack(nn.Module):
    """ A stack of GPT Layers """

    def __init__(self, vocab_size: int, d_model: int, num_blocks: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model     
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = blocks.PositionalEncoding()
        self.layers = nn.ModuleList([GPT(d_model, num_heads, hidden_dim, dropout_rate) for _ in range(num_blocks)])
        self.final_output_logits = nn.Linear(d_model, vocab_size)
        self.norm1 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of tokens: (Batch_size, num_tokens)

        x = self.embedding(x)
        x = self.positional_encoding(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        for layer in self.layers:
            x = layer(x)
        
        norm_output = self.norm1(x)

        logits = self.final_output_logits(norm_output)
        # Dimension: (Batch_size, num_tokens, vocab_size)

        return logits
        

In [ ]:
batch_size = 3
num_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 7
hidden_dim = 16
num_blocks = 6